# Two-Factor Hebbian Baseline, v2: Real Held-Out Test Accuracy

**Why this v2 exists.** The original version of this baseline comparison (Section 5.4) only reported
training-batch accuracy for all three methods -- it never checked generalization to unseen data.
That's a real inconsistency with the rigor used everywhere else in Section 6 (Test 1-3 all evaluate
on proper held-out test sets), so this notebook fixes it: same three methods, same shared model and
training data as before, but now with a genuine MNIST test-set split held out and never trained on,
used only for final evaluation.

**Everything else is unchanged** from the original baseline notebook: same 784-64-64-10 tanh MLP,
same 2,000-sample training subset, same 300 training steps, same hyperparameters for all three
methods (backprop+Adam, three-factor local rule with M=10, two-factor Hebbian). Only the evaluation
protocol changes.

## Step 0 — Setup

In [1]:
import json
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import torchvision
import torchvision.transforms as transforms

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

RESULTS_LOG = []

def log_result(name, config, result, seed):
    entry = {
        'name': name, 'seed': seed, 'config': config, 'result': result,
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    }
    RESULTS_LOG.append(entry)
    print(f"[logged] {name}: {result}")
    return entry

def save_provenance(path='provenance_two_factor_hebbian_v2.json'):
    with open(path, 'w') as f:
        json.dump(RESULTS_LOG, f, indent=2, default=str)
    print(f'Saved {len(RESULTS_LOG)} logged results to {path}')


Using device: cuda


## Step 1 — Model, and a REAL train/test split this time

Same 784-64-64-10 architecture and the same 2,000-sample training subset as the original run, but
now paired with a genuinely held-out MNIST test set (never seen during training, used only for
final evaluation).

In [2]:
INPUT_DIM = 784
HIDDEN = 64
OUTPUT_DIM = 10
N_TRAIN_SAMPLES = 2000
N_TEST_SAMPLES = 2000
BATCH_SIZE = 64
SEED = 0

torch.manual_seed(SEED)
np.random.seed(SEED)

transform = transforms.Compose([transforms.ToTensor()])
full_train = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
full_test = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_idx = torch.randperm(len(full_train))[:N_TRAIN_SAMPLES]
X_train = torch.stack([full_train[i][0].view(-1) for i in train_idx]).to(device)
Y_train = torch.tensor([full_train[i][1] for i in train_idx]).to(device)

# genuinely separate dataset split (torchvision's train=False set), never touched during training
test_idx = torch.randperm(len(full_test))[:N_TEST_SAMPLES]
X_test = torch.stack([full_test[i][0].view(-1) for i in test_idx]).to(device)
Y_test = torch.tensor([full_test[i][1] for i in test_idx]).to(device)

print(f'Training subset: {X_train.shape[0]} samples (MNIST train split)')
print(f'Held-out test set: {X_test.shape[0]} samples (MNIST test split -- never trained on)')

def make_model(seed=SEED):
    torch.manual_seed(seed)
    return nn.Sequential(
        nn.Linear(INPUT_DIM, HIDDEN), nn.Tanh(),
        nn.Linear(HIDDEN, HIDDEN), nn.Tanh(),
        nn.Linear(HIDDEN, OUTPUT_DIM)
    ).to(device)

def get_batch(step, batch_size=BATCH_SIZE):
    n = X_train.shape[0]
    start = (step * batch_size) % n
    idx = torch.arange(start, start + batch_size) % n
    return X_train[idx], Y_train[idx]

@torch.no_grad()
def eval_test_accuracy(model, batch_size=512):
    correct, total = 0, 0
    for i in range(0, X_test.shape[0], batch_size):
        xb, yb = X_test[i:i+batch_size], Y_test[i:i+batch_size]
        out = model(xb)
        correct += (out.argmax(1) == yb).sum().item()
        total += xb.shape[0]
    return correct / total


100%|██████████| 9.91M/9.91M [00:00<00:00, 17.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 490kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.49MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.57MB/s]


Training subset: 2000 samples (MNIST train split)
Held-out test set: 2000 samples (MNIST test split -- never trained on)


## Step 2 — Method A: Backprop (Adam)

In [3]:
def run_backprop(n_steps=300, lr=1e-3, seed=SEED):
    model = make_model(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses, accs = [], []
    for step in range(n_steps):
        x, y = get_batch(step)
        opt.zero_grad(set_to_none=True)
        out = model(x)
        loss = F.cross_entropy(out, y)
        loss.backward()
        opt.step()
        losses.append(loss.item())
        accs.append((out.argmax(1) == y).float().mean().item())
    return model, losses, accs

model_bp, losses_bp, accs_bp = run_backprop()
test_acc_bp = eval_test_accuracy(model_bp)
print(f'Backprop  -- final TRAINING batch loss: {losses_bp[-1]:.4f}, final training batch acc: {accs_bp[-1]:.3f}')
print(f'Backprop  -- HELD-OUT TEST accuracy: {test_acc_bp:.4f}')
log_result('backprop_baseline_v2', {'n_steps': 300, 'lr': 1e-3},
           {'final_train_batch_loss': losses_bp[-1], 'final_train_batch_acc': accs_bp[-1],
            'held_out_test_acc': test_acc_bp}, SEED)


Backprop  -- final TRAINING batch loss: 0.1929, final training batch acc: 0.969
Backprop  -- HELD-OUT TEST accuracy: 0.8840
[logged] backprop_baseline_v2: {'final_train_batch_loss': 0.19293080270290375, 'final_train_batch_acc': 0.96875, 'held_out_test_acc': 0.884}


{'name': 'backprop_baseline_v2',
 'seed': 0,
 'config': {'n_steps': 300, 'lr': 0.001},
 'result': {'final_train_batch_loss': 0.19293080270290375,
  'final_train_batch_acc': 0.96875,
  'held_out_test_acc': 0.884},
 'timestamp': '2026-08-27 21:31:56'}

## Step 3 — Method B: Three-factor local rule

In [4]:
def flat_params(model):
    return torch.cat([p.data.view(-1) for p in model.parameters()])

def set_flat_params(model, flat):
    offset = 0
    for p in model.parameters():
        n = p.numel()
        p.data.copy_(flat[offset:offset+n].view_as(p))
        offset += n

def loss_at(model, flat, x, y):
    set_flat_params(model, flat)
    with torch.no_grad():
        out = model(x)
        return F.cross_entropy(out, y).item(), out

def run_three_factor(n_steps=300, lr=1e-2, sigma=0.01, M=10, seed=SEED):
    model = make_model(seed)
    theta = flat_params(model)
    D = theta.numel()
    losses, accs = [], []
    for step in range(n_steps):
        x, y = get_batch(step)
        g_hat = torch.zeros_like(theta)
        for _ in range(M):
            xi = torch.randn(D, device=device)
            l_plus, _ = loss_at(model, theta + sigma * xi, x, y)
            l_minus, _ = loss_at(model, theta - sigma * xi, x, y)
            g_hat += xi * (l_plus - l_minus) / (2 * sigma)
        g_hat /= M
        theta = theta - lr * g_hat
        step_loss, out = loss_at(model, theta, x, y)
        losses.append(step_loss)
        accs.append((out.argmax(1) == y).float().mean().item())
    set_flat_params(model, theta)
    return model, losses, accs

model_3f, losses_3f, accs_3f = run_three_factor()
test_acc_3f = eval_test_accuracy(model_3f)
print(f'Three-factor -- final TRAINING batch loss: {losses_3f[-1]:.4f}, final training batch acc: {accs_3f[-1]:.3f}')
print(f'Three-factor -- HELD-OUT TEST accuracy: {test_acc_3f:.4f}')
log_result('three_factor_local_rule_v2', {'n_steps': 300, 'lr': 1e-2, 'sigma': 0.01, 'M': 10},
           {'final_train_batch_loss': losses_3f[-1], 'final_train_batch_acc': accs_3f[-1],
            'held_out_test_acc': test_acc_3f}, SEED)


Three-factor -- final TRAINING batch loss: 1.6410, final training batch acc: 0.641
Three-factor -- HELD-OUT TEST accuracy: 0.6190
[logged] three_factor_local_rule_v2: {'final_train_batch_loss': 1.6409971714019775, 'final_train_batch_acc': 0.640625, 'held_out_test_acc': 0.619}


{'name': 'three_factor_local_rule_v2',
 'seed': 0,
 'config': {'n_steps': 300, 'lr': 0.01, 'sigma': 0.01, 'M': 10},
 'result': {'final_train_batch_loss': 1.6409971714019775,
  'final_train_batch_acc': 0.640625,
  'held_out_test_acc': 0.619},
 'timestamp': '2026-08-27 21:32:00'}

## Step 4 — Method C: Two-factor Hebbian (target-blind)

In [5]:
def run_two_factor_hebbian(n_steps=300, eta=1e-4, seed=SEED):
    model = make_model(seed)
    linears = [m for m in model if isinstance(m, nn.Linear)]
    losses, accs = [], []

    def hook_factory(store, key):
        def hook(module, inp, out):
            store[key] = (inp[0].detach(), out.detach())
        return hook

    activity = {}
    handles = [lin.register_forward_hook(hook_factory(activity, i)) for i, lin in enumerate(linears)]

    for step in range(n_steps):
        x, y = get_batch(step)
        with torch.no_grad():
            out = model(x)
            loss = F.cross_entropy(out, y)
        for i, lin in enumerate(linears):
            pre, post = activity[i]
            delta = eta * (post.T @ pre) / pre.shape[0]
            lin.weight.data += delta
        losses.append(loss.item())
        accs.append((out.argmax(1) == y).float().mean().item())

    for h in handles:
        h.remove()
    return model, losses, accs

model_2f, losses_2f, accs_2f = run_two_factor_hebbian()
test_acc_2f = eval_test_accuracy(model_2f)
print(f'Two-factor Hebbian -- initial training acc: {accs_2f[0]:.3f}, final training batch acc: {accs_2f[-1]:.3f}')
print(f'Two-factor Hebbian -- HELD-OUT TEST accuracy: {test_acc_2f:.4f}')
log_result('two_factor_hebbian_baseline_v2', {'n_steps': 300, 'eta': 1e-4},
           {'initial_train_acc': accs_2f[0], 'final_train_batch_acc': accs_2f[-1],
            'held_out_test_acc': test_acc_2f}, SEED)


Two-factor Hebbian -- initial training acc: 0.047, final training batch acc: 0.078
Two-factor Hebbian -- HELD-OUT TEST accuracy: 0.0730
[logged] two_factor_hebbian_baseline_v2: {'initial_train_acc': 0.046875, 'final_train_batch_acc': 0.078125, 'held_out_test_acc': 0.073}


{'name': 'two_factor_hebbian_baseline_v2',
 'seed': 0,
 'config': {'n_steps': 300, 'eta': 0.0001},
 'result': {'initial_train_acc': 0.046875,
  'final_train_batch_acc': 0.078125,
  'held_out_test_acc': 0.073},
 'timestamp': '2026-08-27 21:32:03'}

## Step 5 — Head-to-head table, now with held-out test accuracy

In [6]:
print(f'{"Method":<25}{"Train batch acc":>18}{"HELD-OUT TEST acc":>20}')
print('-' * 65)
print(f'{"Backprop (Adam)":<25}{accs_bp[-1]:>18.3f}{test_acc_bp:>20.4f}')
print(f'{"Three-factor local rule":<25}{accs_3f[-1]:>18.3f}{test_acc_3f:>20.4f}')
print(f'{"Two-factor Hebbian":<25}{accs_2f[-1]:>18.3f}{test_acc_2f:>20.4f}')

log_result('head_to_head_summary_v2', {'n_steps': 300},
           {'backprop_train_acc': accs_bp[-1], 'backprop_test_acc': test_acc_bp,
            'three_factor_train_acc': accs_3f[-1], 'three_factor_test_acc': test_acc_3f,
            'two_factor_train_acc': accs_2f[-1], 'two_factor_test_acc': test_acc_2f}, SEED)


Method                      Train batch acc   HELD-OUT TEST acc
-----------------------------------------------------------------
Backprop (Adam)                       0.969              0.8840
Three-factor local rule               0.641              0.6190
Two-factor Hebbian                    0.078              0.0730
[logged] head_to_head_summary_v2: {'backprop_train_acc': 0.96875, 'backprop_test_acc': 0.884, 'three_factor_train_acc': 0.640625, 'three_factor_test_acc': 0.619, 'two_factor_train_acc': 0.078125, 'two_factor_test_acc': 0.073}


{'name': 'head_to_head_summary_v2',
 'seed': 0,
 'config': {'n_steps': 300},
 'result': {'backprop_train_acc': 0.96875,
  'backprop_test_acc': 0.884,
  'three_factor_train_acc': 0.640625,
  'three_factor_test_acc': 0.619,
  'two_factor_train_acc': 0.078125,
  'two_factor_test_acc': 0.073},
 'timestamp': '2026-08-27 21:32:05'}

## Step 6 — Save provenance

In [7]:
save_provenance('provenance_two_factor_hebbian_v2.json')
print()
print('Contents:')
for entry in RESULTS_LOG:
    print(f"  - {entry['name']}: {entry['result']}")


Saved 4 logged results to provenance_two_factor_hebbian_v2.json

Contents:
  - backprop_baseline_v2: {'final_train_batch_loss': 0.19293080270290375, 'final_train_batch_acc': 0.96875, 'held_out_test_acc': 0.884}
  - three_factor_local_rule_v2: {'final_train_batch_loss': 1.6409971714019775, 'final_train_batch_acc': 0.640625, 'held_out_test_acc': 0.619}
  - two_factor_hebbian_baseline_v2: {'initial_train_acc': 0.046875, 'final_train_batch_acc': 0.078125, 'held_out_test_acc': 0.073}
  - head_to_head_summary_v2: {'backprop_train_acc': 0.96875, 'backprop_test_acc': 0.884, 'three_factor_train_acc': 0.640625, 'three_factor_test_acc': 0.619, 'two_factor_train_acc': 0.078125, 'two_factor_test_acc': 0.073}


## Summary

This closes the one real methodological gap flagged after the original run: all three methods in
Section 5.4's baseline comparison are now evaluated on a genuine held-out MNIST test split, not just
training-batch accuracy. Expect the held-out numbers to be somewhat lower than the training-batch
numbers for all three methods (some overfitting to a 2,000-sample subset is normal) -- report
whatever the real gap turns out to be, rather than assuming it matches the training-batch story
exactly. Once you have these numbers, they should replace the training-batch-only comparison
currently written into Section 6 of the paper.